# Part 8: Virtual Screening
## Screening External Compound Libraries
 
Uses the best trained model to screen large compound libraries
(COCONUT, ZINC) for potential VEGFR2 inhibitors.

In [ ]:
# @title 1. Setup
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("No GPU")

In [ ]:
# @title 2. Load Trained Model
from vegfr2.gnn_pyg import build_pyg_model

# Load best model (GIN with enriched graphs)
model = build_pyg_model('gin', in_dim=2246, hidden=128, layers=3, heads=8, dropout=0.3)

# Load checkpoint
ckpt_path = Path('../Part_4/models/gin/best.pt')
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded model from {ckpt_path}")
else:
    print("No checkpoint found - using random weights (demo)")

model.to(DEVICE).eval()

In [ ]:
# @title 3. Create Screening Function
from vegfr2.features import mol_to_graph_with_fps
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

def screen_compounds(smiles_list, model, device, batch_size=256, threshold=0.5):
    """Screen a list of SMILES and return predictions."""
    data_list = []
    valid_smiles = []
    
    for s in smiles_list:
        try:
            g = mol_to_graph_with_fps(s, use_morgan=True, use_maccs=True)
            data = Data(x=g['node_feats'], edge_index=g['edge_index'],
                       edge_attr=g['edge_feats'],
                       y=torch.tensor([0], dtype=torch.float32))
            data_list.append(data)
            valid_smiles.append(s)
        except:
            continue
    
    if not data_list:
        return pd.DataFrame()
    
    loader = DataLoader(data_list, batch_size=batch_size, shuffle=False)
    
    all_probs = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index, batch.batch)
            probs = torch.sigmoid(logits).squeeze().cpu().numpy()
            all_probs.extend(probs)
    
    results = pd.DataFrame({
        'smiles': valid_smiles,
        'probability': all_probs,
        'predicted_active': [1 if p >= threshold else 0 for p in all_probs]
    })
    
    return results.sort_values('probability', ascending=False)

In [ ]:
# @title 4. Demo Screening with Test Set
test_df = pd.read_csv('data/test.csv')
print(f"Screening {len(test_df)} test compounds...")

results = screen_compounds(test_df['smiles'].tolist(), model, DEVICE)

print(f"\nScreening Results:")
print(f"  Total screened: {len(results)}")
print(f"  Predicted active: {results['predicted_active'].sum()} ({results['predicted_active'].mean():.1%})")
print(f"  Top 10 predictions:")
results.head(10)

In [ ]:
# @title 5. Generate Synthetic Library (Demo)
np.random.seed(42)

# Use test set SMILES as demo library
demo_library = test_df['smiles'].tolist()[:100]

print(f"Demo library: {len(demo_library)} compounds")
print("In practice, use COCONUT or ZINC libraries")

results = screen_compounds(demo_library, model, DEVICE)

In [ ]:
# @title 6. Analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Probability distribution
axes[0].hist(results['probability'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Threshold=0.5')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Screening Probability Distribution')
axes[0].legend()

# Active vs Inactive
active = results[results['predicted_active'] == 1]['probability']
inactive = results[results['predicted_active'] == 0]['probability']
axes[1].hist(inactive, bins=30, alpha=0.7, label='Predicted Inactive', color='blue')
axes[1].hist(active, bins=30, alpha=0.7, label='Predicted Active', color='red')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Active vs Inactive Predictions')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/screening_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# @title 7. Export Hits
hits = results[results['predicted_active'] == 1].copy()
hits.to_csv('data/screening_hits.csv', index=False)
print(f"Exported {len(hits)} hits to data/screening_hits.csv")

# Summary
print("\n" + "=" * 60)
print("VIRTUAL SCREENING SUMMARY")
print("=" * 60)
print(f"  Library size: {len(results)}")
print(f"  Hits (probability >= 0.5): {len(hits)}")
print(f"  Hit rate: {len(hits)/len(results):.1%}")
print(f"  Top candidate: {results.iloc[0]['smiles']}")
print(f"  Top probability: {results.iloc[0]['probability']:.4f}")
print("=" * 60)